In [1]:
import numpy as np
import pandas as pd
from google.cloud import storage

In [2]:
asos_products = pd.read_csv("gs://clip-asos/products_asos.csv")

In [3]:
def remove_entity(text):
  try:
    text_words = text.split(" ")
    idx_to_start = 0
    for w in text_words:
      if w.lower() == w and w!="&":
        break
      idx_to_start += 1
    return (" ").join(text_words[idx_to_start:])
  except:
    return None


def retrieve_first_im(im_list):
  try:
    return json.loads(im_list.replace("'", '"'))[0]
  except:
    return None

import json
asos_products["text"] = asos_products["name"].apply(remove_entity)
asos_products["im"] = asos_products["images"].apply(retrieve_first_im)

In [4]:
asos_text_im = asos_products[["text", "im"]]
asos_text_im = asos_text_im.dropna().drop_duplicates()
train_idx = np.random.choice(asos_text_im.index, size=int(0.8 * len(asos_text_im)), replace=False)
train_ds = asos_text_im.loc[train_idx]
val_ds = asos_text_im[~asos_text_im.index.isin(train_idx)]

In [5]:
train_ds.to_csv("train.csv", index=False)
val_ds.to_csv("val.csv", index=False)

In [8]:
BUCKET_NAME = "clip-asos"
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
bucket.blob("train.csv").upload_from_filename("train.csv")
bucket.blob("val.csv").upload_from_filename("val.csv")